<a href="https://colab.research.google.com/github/Agastyasingh1106/Badminton-AI/blob/talekar-cha-model/badmintonML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Install and Import Ultralytics YOLO
First, we need to install the `ultralytics` package to get access to pre-trained YOLO models (like YOLOv8 or YOLOv11).

In [ ]:
!pip install ultralytics -q

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 6.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


### 2. Define Court Regions
We can define a helper function to map a shuttlecock coordinate `(x, y)` to one of the six regions, assuming normalized coordinates relative to the court boundary dimensions.

* **Top Left** (Region 0)
* **Top Middle** (Region 1)
* **Top Right** (Region 2)
* **Lower Left** (Region 3)
* **Lower Middle** (Region 4)
* **Lower Right** (Region 5)

In [ ]:
def get_court_region(x, y, court_width=100, court_height=100):
    """
    Determine the court region based on 2D coordinates.
    Assumes (0,0) is top-left of the court.
    """
    # Determine vertical position (Top vs Lower)
    is_top = y < (court_height / 2)

    # Determine horizontal position (Left vs Middle vs Right)
    col_width = court_width / 3
    if x < col_width:
        col = "left"
    elif x < 2 * col_width:
        col = "middle"
    else:
        col = "right"

    row = "top" if is_top else "lower"
    return f"{row}_{col}"

# Test mapping
print("Coordinate (15, 25) maps to:", get_court_region(15, 25))
print("Coordinate (50, 75) maps to:", get_court_region(50, 75))

Coordinate (15, 25) maps to: top_left
Coordinate (50, 75) maps to: lower_middle


### 3. Load YOLO Model & Example Training Pipeline
If you have a labeled dataset in YOLO format (containing images of badminton courts with shuttlecocks labeled), you can train the model as shown below.

In [ ]:
# Load a pre-trained Nano model (suitable for fast training and lightweight deployment)
model = YOLO('yolov8n.pt')

# To train the model, you would point to your dataset configuration YAML file:
# results = model.train(data='shuttlecock_dataset.yaml', epochs=50, imgsz=640)
print("YOLO model loaded and ready for training setup!")

YOLO model loaded and ready for training setup!


### 4. Generate Synthetic Coordinate Dataset
Let's generate simulated coordinates of a shuttlecock's current trajectory `(x, y)` and its final landing region. We will train a simple classifier (like a Random Forest or Feedforward Neural Network) on these coordinate pairs to predict the region.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Generate 1000 synthetic coordinate records
np.random.seed(42)
n_samples = 1000

# Assuming standard court size 100x100
X_coords = np.random.uniform(0, 100, size=(n_samples, 2))

# Calculate target labels (6 regions)
y_labels = np.array([get_court_region(x, y) for x, y in X_coords])

# Convert to DataFrame
df = pd.DataFrame(X_coords, columns=['x_position', 'y_position'])
df['landing_region'] = y_labels

# Display some generated data
display(df.head())

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    df[['x_position', 'y_position']], df['landing_region'], test_size=0.2, random_state=42
)

# Train a classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate the model
y_pred = clf.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))

,x_position,y_position,landing_region
0,37.454012,95.071431,lower_middle
1,73.199394,59.865848,lower_right
2,15.601864,15.599452,top_left
3,5.808361,86.617615,lower_left
4,60.111501,70.807258,lower_middle


Classification Report:
               precision    recall  f1-score   support

  lower_left       1.00      1.00      1.00        33
lower_middle       1.00      1.00      1.00        27
 lower_right       1.00      1.00      1.00        37
    top_left       1.00      1.00      1.00        33
  top_middle       1.00      1.00      1.00        38
   top_right       1.00      1.00      1.00        32

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



### 5. Generate Synthetic Image Dataset for YOLO Training
If you want to use a vision-based YOLO model, we can generate synthetic images of a court with a shuttlecock drawn at random locations, along with corresponding YOLO annotations (`.txt` files).

In [ ]:
import os

# Create directories for the synthetic YOLO dataset
os.makedirs('synthetic_dataset/images/train', exist_ok=True)
os.makedirs('synthetic_dataset/labels/train', exist_ok=True)

# Generate synthetic court images containing a shuttlecock
def generate_synthetic_yolo_data(num_images=20):
    for i in range(num_images):
        # 1. Create a green image representing the court (640x640)
        img = np.zeros((640, 640, 3), dtype=np.uint8)
        img[:] = [34, 139, 34] # Forest green

        # Draw court boundary lines (white)
        cv2.rectangle(img, (50, 50), (590, 590), (255, 255, 255), 3)

        # Random shuttlecock position
        shuttle_x = np.random.randint(70, 570)
        shuttle_y = np.random.randint(70, 570)

        # Draw the shuttlecock (a small white circle with a tip)
        cv2.circle(img, (shuttle_x, shuttle_y), 10, (255, 255, 255), -1)
        cv2.line(img, (shuttle_x, shuttle_y), (shuttle_x, shuttle_y - 15), (200, 200, 200), 2)

        # Save image
        img_path = f'synthetic_dataset/images/train/court_{i}.jpg'
        cv2.imwrite(img_path, img)

        # 2. Save matching YOLO label (normalized coordinates)
        # YOLO format: class_id x_center y_center width height
        class_id = 0 # 0: shuttlecock
        x_center = shuttle_x / 640.0
        y_center = shuttle_y / 640.0
        w = 25.0 / 640.0
        h = 25.0 / 640.0

        label_path = f'synthetic_dataset/labels/train/court_{i}.txt'
        with open(label_path, 'w') as f:
            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

generate_synthetic_yolo_data()
print("Generated synthetic YOLO training images and labels successfully!")

Generated synthetic YOLO training images and labels successfully!


### 6. Create Dataset Configuration and Train YOLO Model
We will write a dataset configuration YAML file that tells YOLO where to find the training images and what classes to expect (0: shuttlecock). Then, we will initiate training.

In [ ]:
import yaml

# Define the dataset configuration
dataset_config = {
    'path': '/content/synthetic_dataset',
    'train': 'images/train',
    'val': 'images/train',
    'names': {
        0: 'shuttlecock'
    }
}

# Save configuration to a YAML file
with open('shuttlecock_dataset.yaml', 'w') as f:
    yaml.dump(dataset_config, f)

print("Created shuttlecock_dataset.yaml!")

# Train the YOLOv8 model on our synthetic images for 3 epochs (fast demonstration)
results = model.train(data='shuttlecock_dataset.yaml', epochs=3, imgsz=640)

Created shuttlecock_dataset.yaml!
Ultralytics 8.4.183 🚀 Python-3.13.16 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=shuttlecock_dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optim

### 7. Run Inference & Predict Court Landing Region
Once trained, we can detect a shuttlecock from a new image, extract its coordinates, and determine its corresponding region on the court.

In [ ]:
# Load the trained model weights
trained_model = YOLO('/content/runs/detect/train/weights/best.pt')

# Run prediction on one of our synthetic images
prediction_results = trained_model('synthetic_dataset/images/train/court_0.jpg')

# Process prediction and map coordinate to region
for res in prediction_results:
    boxes = res.boxes
    for box in boxes:
        # Get bounding box coordinates
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

        # Find center position of the shuttlecock
        shuttle_x = (x1 + x2) / 2
        shuttle_y = (y1 + y2) / 2

        # Map pixels (0 to 640) to court coordinate bounds (0 to 100)
        mapped_x = (shuttle_x / 640.0) * 100
        mapped_y = (shuttle_y / 640.0) * 100

        region = get_court_region(mapped_x, mapped_y)
        print(f"Detected shuttlecock at Pixel ({shuttle_x:.1f}, {shuttle_y:.1f})")
        print(f"Mapped Court Coordinates: ({mapped_x:.1f}, {mapped_y:.1f})")
        print(f"Predicted Landing Region: {region}")


image 1/1 /content/synthetic_dataset/images/train/court_0.jpg: 640x640 1 shuttlecock, 186.7ms
Speed: 4.9ms preprocess, 186.7ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 640)
Detected shuttlecock at Pixel (289.8, 458.4)
Mapped Court Coordinates: (45.3, 71.6)
Predicted Landing Region: lower_middle


### 6. Create Dataset Configuration and Train YOLO Model
We will write a dataset configuration YAML file that tells YOLO where to find the training images and what classes to expect (0: shuttlecock). Then, we will initiate training.

In [ ]:
import yaml

# Define the dataset configuration
dataset_config = {
    'path': '/content/synthetic_dataset', # Dataset root directory
    'train': 'images/train',             # Training images
    'val': 'images/train',               # Using train as validation for demonstration
    'names': {
        0: 'shuttlecock'
    }
}

# Save configuration to a YAML file
with open('shuttlecock_dataset.yaml', 'w') as f:
    yaml.dump(dataset_config, f)

print("Created shuttlecock_dataset.yaml!")

# Train the YOLOv8 model on our synthetic images for 3 epochs (fast demonstration)
results = model.train(data='shuttlecock_dataset.yaml', epochs=3, imgsz=640)

Created shuttlecock_dataset.yaml!
Ultralytics 8.4.183 🚀 Python-3.13.16 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=shuttlecock_dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/train/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, n

### 7. Run Inference & Predict Court Landing Region
Once trained, we can detect a shuttlecock from a new image, extract its coordinates, and determine its corresponding region on the court.

In [ ]:
# Load the trained model weights
trained_model = YOLO('/content/runs/detect/train/weights/best.pt')

# Run prediction on one of our synthetic images
prediction_results = trained_model('synthetic_dataset/images/train/court_0.jpg')

# Process prediction and map coordinate to region
for res in prediction_results:
    boxes = res.boxes
    for box in boxes:
        # Get bounding box coordinates
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

        # Find center position of the shuttlecock
        shuttle_x = (x1 + x2) / 2
        shuttle_y = (y1 + y2) / 2

        # Map pixels (0 to 640) to court coordinate bounds (0 to 100)
        mapped_x = (shuttle_x / 640.0) * 100
        mapped_y = (shuttle_y / 640.0) * 100

        region = get_court_region(mapped_x, mapped_y)
        print(f"Detected shuttlecock at Pixel ({shuttle_x:.1f}, {shuttle_y:.1f})")
        print(f"Mapped Court Coordinates: ({mapped_x:.1f}, {mapped_y:.1f})")
        print(f"Predicted Landing Region: {region}")


image 1/1 /content/synthetic_dataset/images/train/court_0.jpg: 640x640 1 shuttlecock, 128.1ms
Speed: 2.8ms preprocess, 128.1ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)
Detected shuttlecock at Pixel (289.8, 458.4)
Mapped Court Coordinates: (45.3, 71.6)
Predicted Landing Region: lower_middle


### 8. Process Video Data and Predict Court Regions
This script reads an input video file, detects the shuttlecock, overlays the predicted court region onto the video frames, and saves the annotated video.

In [ ]:
def process_court_video(input_video_path, output_video_path):
    # Load the trained model
    trained_model = YOLO('/content/runs/detect/train/weights/best.pt')

    # Open the video file
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print("Error: Could not open video file.")
        return

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) if hasattr(cv2, 'CAP_PROP_FRAME_WIDTH') else cap.get(3))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) if hasattr(cv2, 'CAP_PROP_FRAME_HEIGHT') else cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS if hasattr(cv2, 'CAP_PROP_FPS') else 5)

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    print("Processing video frames...")
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Run prediction on the current frame
        results = trained_model(frame, verbose=False)

        # Process detections
        for res in results:
            boxes = res.boxes
            for box in boxes:
                # Get bounding box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

                # Find center position of the shuttlecock
                shuttle_x = (x1 + x2) / 2
                shuttle_y = (y1 + y2) / 2

                # Map pixels to court coordinate bounds (0 to 100)
                mapped_x = (shuttle_x / width) * 100
                mapped_y = (shuttle_y / height) * 100

                region = get_court_region(mapped_x, mapped_y)

                # Draw bounding box and landing region info on the frame
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 2)
                cv2.putText(frame, f"Region: {region}", (int(x1), int(y1) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        # Write the processed frame to output video
        out.write(frame)
        frame_count += 1

    cap.release()
    out.release()
    print(f"Finished processing! Saved annotated video to: {output_video_path}")

### 9. Inspect Uploaded CSV Structure
Let's load and display the contents of one of the uploaded `.csv` files to understand its structure, columns, and coordinates.

In [ ]:
import glob

# Find all uploaded CSV files
csv_files = sorted(glob.glob('/content/*_predict.csv'))

if csv_files:
    test_csv_path = csv_files[0]
    print(f"Inspecting file: {test_csv_path}")
    df_test = pd.read_csv(test_csv_path)
    display(df_test.head(10))
    print("Columns present in dataset:", df_test.columns.tolist())
else:
    print("No matching CSV files found in /content/")

Inspecting file: /content/00376_predict.csv


,Frame,Visibility,X,Y,Time
0,2,0,0,0,00:00:00.06666666666666667
1,3,0,0,0,00:00:00.1
2,4,0,0,0,00:00:00.13333333333333333
3,5,0,0,0,00:00:00.16666666666666666
4,6,0,0,0,00:00:00.2
5,7,0,0,0,00:00:00.23333333333333334
6,8,0,0,0,00:00:00.26666666666666666
7,9,0,0,0,00:00:00.3
8,10,0,0,0,00:00:00.3333333333333333
9,11,0,0,0,00:00:00.36666666666666664


Columns present in dataset: ['Frame', 'Visibility', 'X', 'Y', 'Time']


### 10. Process Uploaded Videos & CSVs into a YOLO Dataset
This script will map the `(X, Y)` pixels from each CSV to its corresponding `.mp4` video frame, construct the bounding boxes, and save them in the structure required for YOLO training.

In [ ]:
import os
import glob
import pandas as pd
import cv2

# Create directory paths
os.makedirs('/content/real_dataset/images/train', exist_ok=True)
os.makedirs('/content/real_dataset/labels/train', exist_ok=True)
os.makedirs('/content/real_dataset/images/val', exist_ok=True)
os.makedirs('/content/real_dataset/labels/val', exist_ok=True)

# Find all matching base names
all_csvs = sorted(glob.glob('/content/*_predict.csv'))

print(f"Found {len(all_csvs)} matching video-CSV pairs. Extracting frames...")

sample_limit_per_video = 15 # Extract up to 15 active frames per file to train efficiently
train_split = 0.8

for index, csv_path in enumerate(all_csvs):
    base_name = os.path.basename(csv_path).replace('_predict.csv', '')
    video_path = f'/content/{base_name}_predict.mp4'

    if not os.path.exists(video_path):
        continue

    # Decide partition
    split_dir = 'train' if (index / len(all_csvs)) < train_split else 'val'

    # Load tracks
    df_track = pd.read_csv(csv_path)
    active_tracks = df_track[df_track['Visibility'] == 1].head(sample_limit_per_video)

    if active_tracks.empty:
        continue

    cap = cv2.VideoCapture(video_path)

    for idx, row in active_tracks.iterrows():
        frame_id = int(row['Frame'])
        x, y = int(row['X']), int(row['Y'])

        # Set capture pointer to the specific frame
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()
        if not ret:
            continue

        h, w, _ = frame.shape

        # Save frame image
        img_name = f"{base_name}_frame_{frame_id}.jpg"
        img_save_path = f"/content/real_dataset/images/{split_dir}/{img_name}"
        cv2.imwrite(img_save_path, frame)

        # Create bounding box (e.g. 20x20 pixels around center)
        box_w, box_h = 24.0, 24.0
        x_center = x / w
        y_center = y / h
        norm_w = box_w / w
        norm_h = box_h / h

        # Save YOLO annotation file
        label_name = f"{base_name}_frame_{frame_id}.txt"
        label_save_path = f"/content/real_dataset/labels/{split_dir}/{label_name}"
        with open(label_save_path, 'w') as f:
            f.write(f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")

    cap.release()

print("Real dataset generation and preparation complete!")

Found 25 matching video-CSV pairs. Extracting frames...
Real dataset generation and preparation complete!


### 11. Train YOLO Model on Your Uploaded Dataset
Now we configure the model to train on the real annotated dataset we generated from your uploaded tracks.

In [ ]:
# Save the real dataset YAML configuration
real_dataset_config = {
    'path': '/content/real_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'shuttlecock'
    }
}

with open('real_shuttlecock_dataset.yaml', 'w') as f:
    yaml.dump(real_dataset_config, f)

# Train our model on the real dataset
real_results = model.train(data='real_shuttlecock_dataset.yaml', epochs=5, imgsz=640)

Ultralytics 8.4.183 🚀 Python-3.13.16 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=real_shuttlecock_dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/train-2/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=None, opset=None